# RAG Agent -- Process Check

- Create a LLM client
- RAG Prep: register OpenAIEmbeddings, pdf_path, pdf_loader, text_splitter 
- Register page split and the rest of RAG configurations including persist_directory, collection_name
- Make a vectorstore and retriever
- Make a retriever_tool and configure the tool settings
- Set langchain agent state
- Make conditional edge function
- Create the main llm agent fucntion
- Make a retriever agent that acts as manual ToolNode
- Register graph order
- Create the execution function

## RAG Workflow: 
LLM

↓

call_llm()

↓

LLM outputs tool_call

↓

should_continue → True

↓

take_action()

↓

tools_dict["retriever_tool"].invoke(query)

↓

retriever_tool(query)

↓

retriever.invoke(query)

↓

ChromaDB

In [4]:
from dotenv import load_dotenv
import os
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings 
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.tools import tool

In [5]:
load_dotenv()

True

In [12]:
llm = ChatOpenAI(model="gpt-5-mini")

## ChromaDB RAG Config

In [21]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

pdf_path = "Stock_Market_Performance_2024.pdf"

if not os.path.exists(pdf_path): 
    FileNotFoundError(f"PDF file not found: {pdf_path}")

pages = PyPDFLoader(pdf_path).load()

pdf_loader = PyPDFLoader(pdf_path)

try:
    pages = pdf_loader.load()
    print(f"PDF file was loaded -- {len(pages)} pages")
except Exception as e:
    print(f"Error loading PDF: {e}")
    raise

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

pages_split = text_splitter.split_documents(pages)
persist_directory = r"/Volumes/VTG/Dev/LangGraph tutorial"
collection_name = "stock_market"

if not os.path.exists(persist_directory):
    os.makedirs(persist_directory)

PDF file was loaded -- 9 pages


In [24]:
try:
    vectorstore = Chroma.from_documents(
        documents=pages_split,
        embedding=embeddings, 
        persist_directory=persist_directory,
        collection_name=collection_name
    )
    print("Vector store has been created!")
except Exception as e:
    print(f"Error creating vectorstore: {str(e)}")
    raise

Vector store has been created!


In [25]:
retriever = vectorstore.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 5}
)

## Tool Function

In [26]:
@tool
def retriever_tool(query: str) -> str:
    """
    This tool searches and returns the information from the Stock Market Performance 2024 document.
    """

    docs = retriever.invoke(query)
    
    if not docs:
        return "I found no relevant information..."

    results = []
    for i, doc in enumerate(docs):
        results.append(f"Document {i + 1}:\n{doc.page_content}")

    return "\n\n".join(results)
    